# SAIGE NULL MODEL
The following script is used to fit the null model for the set-based SAIGE-GENE analysis. The options useSparseGRMtoFitNULL, useSparseGRMforVarRatio, and isCateVarianceRatio must be set to TRUE. This script uses the sparse GRM, 2,000 randomly selected variants in the specified MAC categories, and the phenotype file with covariates to fit the null model.

In [ ]:
# Load packages 
import os

# Set environment variable for workspace bucket 
my_bucket = 'gs://rw-migration-aou-rw-9f879fc4'
%env WORKSPACE_BUCKET=gs://rw-migration-aou-rw-9f879fc4
%env ARTIFACT_REGISTRY_DOCKER_REPO=us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod
%run /home/jupyter/workspace/rw-migration-aou-rw-9f879fc4/notebooks/Setting_Env_Variables.ipynb

# COMMAND 

In [ ]:
%%writefile saige_gene_step01.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    
    echo "Processing population: ${pop}"
    
    # Loop through the phenotypes
    for pheno in aml_covariates; do 

        # Loop through the covariates
        for cov in age_squared age_sex age_biospecimen biospecimen; do 

            echo "Processing population: ${pop} | phenotype: ${pheno} | covariate: ${cov}"

            step1_fitNULLGLMM.R \
                --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
                --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
                --plinkFile="${variance_files}/${pop}_array.forCate_vr" \
                --phenoFile="${phenotype}/${pheno}_files/${pop}_phenotype_file.txt" \
                --useSparseGRMtoFitNULL=TRUE \
                --useSparseGRMforVarRatio=TRUE \
                --IsOverwriteVarianceRatioFile=TRUE \
                --isCateVarianceRatio=TRUE \
                --phenoCol=aml_phenotype \
                --traitType=binary \
                --covarColList="age,sex,${cov},PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10" \
                --qCovarColList=sex \
                --sampleIDColinphenoFile=person_id \
                --outputPrefix="${output}/${pop}_${pheno}_${cov}_null_glmm" 

        done 
    done
done


In [ ]:
# Save script to workspace bucket 
!gcloud storage cp saige_gene_step01.sh {my_bucket}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gcloud storage cat {my_bucket}/data/scripts/saige_gene_step01.sh 

# SUBMISSION

In [ ]:
%%bash --out saige_gene_step01

# Source the dsub configuration file
source ~/aou_dsub.bash 

# Command for dsub submission
aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --preemptible \
  --machine-type n1-standard-1 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input-recursive phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --env phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files" \
  --env grm="${WORKSPACE_BUCKET}/data/saige_step00/grm" \
  --env variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files" \
  --env output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm" \
  --script "${WORKSPACE_BUCKET}/data/scripts/saige_gene_step01.sh"
